In [ ]:
import glob
import json
import os
import pickle
import re
from typing import Dict, List
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

# 6 個感測器的硬體測試項目定義
SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

# 不適合作為預測特徵的測試後識別資訊與開關資訊
METADATA_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]


def sanitize_column_name(col_name: str) -> str:
    """去除 LightGBM C++ 核心不支援的 JSON 特殊字元"""
    return re.sub(r"[\[\]\{\}:\",]", "_", col_name)


def load_wafer_file(filepath: str) -> pd.DataFrame:
    """讀取 Advantest 原始 CSV，乾淨略過前 4 行定義規格 (Pin, TestNum, High/Low Limit)"""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)
    df.columns = [sanitize_column_name(c) for c in df.columns]

    # 將所有測試項轉換為純數值型別 (float)
    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def get_causal_features(
    all_raw_cols: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """嚴格依照時序提取 target_col 之前的所有測試項目 (保證零數據外洩)"""
    sanitized_raw = [sanitize_column_name(c) for c in all_raw_cols]
    sanitized_target = sanitize_column_name(target_col)
    target_idx = sanitized_raw.index(sanitized_target)

    # 欄位 10 之前為晶片元數據，切取 10 到目標測試項之間的量測特徵
    candidate_features = sanitized_raw[10:target_idx]
    sanitized_targets = [
        sanitize_column_name(s["col"]) for s in SENSOR_TARGETS
    ]

    features = [
        c
        for c in candidate_features
        if c not in METADATA_COLS and c not in sanitized_targets
    ]

    # 加入晶粒實體坐標與量測站點
    spatial_features = [c for c in ["Site", "X", "Y"] if c in df.columns]
    selected = spatial_features + features

    return list(dict.fromkeys(selected))


def get_model():
    """統一、穩健且具備泛化防禦力的 LightGBM 結構"""
    return lgb.LGBMRegressor(
        n_estimators=100,
        max_depth=4,
        num_leaves=10,
        min_child_samples=15,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_alpha=0.05,
        reg_lambda=2.0,
        objective="regression",
        n_jobs=-1,
        random_state=42,
        verbosity=-1,
    )


# =========================================================================
# 驗證模式：5-Fold 晶圓級隔離交叉驗證 (Leave-Wafer-Out CV)
# =========================================================================
def run_validation(data_dir: str):
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        print(f"錯誤：在目錄 '{data_dir}' 中找不到 *RawResult.csv 檔案。")
        return

    print(f"載入 {len(filepaths)} 片晶圓數據進行 5-Fold 驗證...")
    dfs = [load_wafer_file(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    print("\n" + "=" * 75)
    print(" 5-FOLD WAFER GROUP CROSS-VALIDATION (CLEAN RAW DATA)")
    print("=" * 75)

    all_stage_mae = []

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        # 1. 取得因果時序特徵
        feature_cols = get_causal_features(raw_columns, target_col, df_all)

        # 2. 允許使用在該 Sensor 之前已經量測完畢的 Sensor 真值
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_all.columns
            ):
                feature_cols.append(prev_col)

        # 3. Top-80 特徵精簡 (當前測試項超過 80 個時執行，消除高維噪訊)
        X = df_all[feature_cols].copy()
        y = df_all[target_col].copy()
        valid_idx = y.dropna().index

        if len(feature_cols) > 80:
            quick_fit = lgb.LGBMRegressor(
                n_estimators=30, random_state=42, verbosity=-1, n_jobs=-1
            )
            quick_fit.fit(X.loc[valid_idx], y.loc[valid_idx])
            importances = pd.Series(
                quick_fit.feature_importances_, index=feature_cols
            )
            feature_cols = (
                importances.sort_values(ascending=False).head(80).index.tolist()
            )
            X = X[feature_cols]

        # 4. 執行晶圓級盲測驗證
        fold_maes, fold_r2s = [], []
        for train_idx, val_idx in gkf.split(X, y, wafers):
            X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
            X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

            v_tr = y_tr.dropna().index
            v_va = y_va.dropna().index

            model = get_model()
            model.fit(X_tr.loc[v_tr], y_tr.loc[v_tr])

            preds = model.predict(X_va.loc[v_va])
            fold_maes.append(mean_absolute_error(y_va.loc[v_va], preds))
            fold_r2s.append(r2_score(y_va.loc[v_va], preds))

        stage_mae = np.mean(fold_maes)
        stage_r2 = np.mean(fold_r2s)
        all_stage_mae.append(stage_mae)

        print(
            f"Stage {s_idx} ({target_col:<22}) | Feats: {len(feature_cols):>2} | 5-Fold MAE: {stage_mae:.4f} °C | R2: {stage_r2:.4f}"
        )

    print("-" * 75)
    print(f"全階段平均跨晶圓 MAE: {np.mean(all_stage_mae):.4f} °C")


# =========================================================================
# 生產模式：使用全量 25 片晶圓重訓並匯出最終模型
# =========================================================================
def train_and_export_all(data_dir: str = "./", output_dir: str = "./models"):
    os.makedirs(output_dir, exist_ok=True)
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        print(f"錯誤：在目錄 '{data_dir}' 中找不到 *RawResult.csv 檔案。")
        return

    print("\n" + "=" * 75)
    print(
        f" 正式生產訓練：使用全部 {len(filepaths)} 片晶圓 (全量 2,000 顆 Die)"
    )
    print("=" * 75)

    dfs = [load_wafer_file(fp) for fp in filepaths]
    df_full = pd.concat(dfs, ignore_index=True)
    raw_columns = list(pd.read_csv(filepaths[0], nrows=1).columns)

    feature_schemas: Dict[int, List[str]] = {}

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = sanitize_column_name(sensor["col"])

        # 1. 因果時序切片
        feature_cols = get_causal_features(raw_columns, target_col, df_full)
        for prev in SENSOR_TARGETS:
            prev_col = sanitize_column_name(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in feature_cols
                and prev_col in df_full.columns
            ):
                feature_cols.append(prev_col)

        X = df_full[feature_cols].copy()
        y = df_full[target_col].copy()
        valid_idx = y.dropna().index

        # 2. 特徵精簡 (若 > 80 維則精簡)
        if len(feature_cols) > 80:
            quick_fit = lgb.LGBMRegressor(
                n_estimators=30, random_state=42, verbosity=-1, n_jobs=-1
            )
            quick_fit.fit(X.loc[valid_idx], y.loc[valid_idx])
            importances = pd.Series(
                quick_fit.feature_importances_, index=feature_cols
            )
            feature_cols = (
                importances.sort_values(ascending=False).head(80).index.tolist()
            )
            X = X[feature_cols]

        feature_schemas[s_idx] = feature_cols

        # 3. 正式訓練並保存
        model = get_model()
        model.fit(X.loc[valid_idx], y.loc[valid_idx])

        model_path = os.path.join(output_dir, f"model_sensor{s_idx}.pkl")
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        print(
            f"  [✓] Sensor {s_idx} 模型已儲存 -> 使用 {len(feature_cols):>2} 個關鍵特徵"
        )

    # 4. 導出特徵欄位對照表供 Docker sample.py 讀取
    schema_path = os.path.join(output_dir, "feature_schema.json")
    with open(schema_path, "w") as f:
        json.dump(feature_schemas, f, indent=2)

    print(f"\n成功匯出所有 6 個模型與特徵對照表至: '{output_dir}/'")


# =========================================================================
# VS Code 入口點 (直接按 Run 執行)
# =========================================================================
if __name__ == "__main__":
    # 執行模式選擇:
    #   "VALIDATE"  -> 跑 5-Fold 晶圓盲測驗證，確認真實 MAE/R2 指標
    #   "EXPORT"    -> 使用全部 25 片晶圓重訓並導出 .pkl 與 .json 到 ./models/
    #   "ALL"       -> 先驗證，接著導出正式模型
    MODE = "ALL"

    DATA_DIR = "./"  # 存放 25 片 CSV 的資料夾路徑
    OUTPUT_DIR = "./models"

    if MODE in ("VALIDATE", "ALL"):
        run_validation(DATA_DIR)

    if MODE in ("EXPORT", "ALL"):
        train_and_export_all(DATA_DIR, OUTPUT_DIR)

In [6]:
import glob
import json
import os
import pickle
import re
from typing import Dict, List, Tuple
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

# ==============================================================================
# 1. 全域硬體時序與參數配置 (Hardware Flow & Configurations)
# ==============================================================================

SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

# 不可作為特徵的非參數測試欄位
EXCLUDED_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]

# 5-Fold Group 盲測驗證收斂的最佳超參數矩陣
BEST_HYPERPARAMS = {
    1: {
        "subsample": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.02,
        "objective": "regression",
        "num_leaves": 7,
        "n_estimators": 150,
        "min_child_samples": 10,
        "max_depth": 4,
        "learning_rate": 0.1,
        "colsample_bytree": 0.6,
    },
    2: {
        "subsample": 0.8,
        "reg_lambda": 6.0,
        "reg_alpha": 0.0,
        "objective": "huber",
        "num_leaves": 7,
        "n_estimators": 150,
        "min_child_samples": 15,
        "max_depth": 6,
        "learning_rate": 0.1,
        "colsample_bytree": 0.6,
    },
    3: {
        "subsample": 0.8,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "objective": "regression",
        "num_leaves": 7,
        "n_estimators": 90,
        "min_child_samples": 10,
        "max_depth": 3,
        "learning_rate": 0.14,
        "colsample_bytree": 1.0,
    },
    4: {
        "subsample": 0.8,
        "reg_lambda": 0.5,
        "reg_alpha": 0.05,
        "objective": "regression",
        "num_leaves": 10,
        "n_estimators": 150,
        "min_child_samples": 15,
        "max_depth": 6,
        "learning_rate": 0.1,
        "colsample_bytree": 0.8,
    },
    5: {
        "subsample": 0.8,
        "reg_lambda": 0.5,
        "reg_alpha": 0.05,
        "objective": "regression",
        "num_leaves": 10,
        "n_estimators": 150,
        "min_child_samples": 15,
        "max_depth": 6,
        "learning_rate": 0.1,
        "colsample_bytree": 0.8,
    },
    6: {
        "subsample": 0.8,
        "reg_lambda": 0.5,
        "reg_alpha": 0.05,
        "objective": "regression",
        "num_leaves": 10,
        "n_estimators": 150,
        "min_child_samples": 15,
        "max_depth": 6,
        "learning_rate": 0.1,
        "colsample_bytree": 0.8,
    },
}
# ==============================================================================
# 2. 資料清洗與因果特徵提取模組 (Data Processing & Causality Boundary)
# ==============================================================================


def clean_col(name: str) -> str:
    """清理 LightGBM C++ 不支援的字元"""
    return re.sub(r"[\[\]\{\}:\",]", "_", name)


def read_wafer_csv(filepath: str) -> pd.DataFrame:
    """讀取單片晶圓 CSV，跳過規格定義行，並建立 Touchdown 空間與物理指標"""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)
    df.columns = [clean_col(c) for c in df.columns]

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 1. 晶粒在承載盤的推進進度 (Touchdown Index: 0..19)
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # 2. 靜態漏電流對數物理轉換 (Log Subthreshold Leakage)
    for ic in [c for c in df.columns if "IDDQ" in c]:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    # 3. 同 Touchdown 跨 4 個 Site 的基準中心偏置
    key_iddq = clean_col("80000_Main.IDDQ_flow.IDDQ_A1#IO1")
    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    return df


def get_causal_columns(
    raw_header: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """嚴格防止未來數據外洩：只提取在該 Sensor 呼叫前已經執行的測試項"""
    clean_raw = [clean_col(c) for c in raw_header]
    target_idx = clean_raw.index(target_col)

    # 提取物理機台時序在此測試項前面的欄位
    pre_tests = clean_raw[10:target_idx]
    sensor_names = [clean_col(s["col"]) for s in SENSOR_TARGETS]

    base_features = [
        c
        for c in pre_tests
        if c not in EXCLUDED_COLS and c not in sensor_names and c in df.columns
    ]

    # 加入坐標與物理特徵
    context_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
    ]
    log_iddq = [c for c in df.columns if c.startswith("log_IDDQ")]

    all_candidates = [
        c
        for c in (context_features + log_iddq + base_features)
        if c in df.columns
    ]
    return list(dict.fromkeys(all_candidates))


# ==============================================================================
# 3. 5-Fold Leave-Wafer-Out 交叉驗證 (Validation Mode)
# ==============================================================================


def run_5fold_validation(data_dir: str):
    """執行 5-Fold Group 跨晶圓盲測驗證，確認真實泛化力"""
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        print(f"錯誤：在目錄 '{data_dir}' 找不到 *RawResult.csv 檔案！")
        return

    print(f"\n載入 {len(filepaths)} 片晶圓進行 5-Fold 跨晶圓盲測...")
    dfs = [read_wafer_csv(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_header = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    print("\n" + "=" * 75)
    print(" 5-FOLD WAFER GROUP VALIDATION RESULTS (TOP-80 PRUNED)")
    print("=" * 75)

    all_stage_maes = []

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = clean_col(sensor["col"])

        features = get_causal_columns(raw_header, target_col, df_all)
        # 允許加入時序在前的真實感測器數值
        for prev in SENSOR_TARGETS:
            prev_col = clean_col(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in features
                and prev_col in df_all.columns
            ):
                features.append(prev_col)

        X = df_all[features].copy()
        y = df_all[target_col].copy()
        valid_idx = y.dropna().index

        # Top-80 特徵篩選
        if len(features) > 80:
            selector = lgb.LGBMRegressor(
                n_estimators=40, random_state=42, verbosity=-1, n_jobs=-1
            )
            selector.fit(X.loc[valid_idx], y.loc[valid_idx])
            top_feats = (
                pd.Series(selector.feature_importances_, index=features)
                .sort_values(ascending=False)
                .head(80)
                .index.tolist()
            )
            X = X[top_feats]
            used_cnt = len(top_feats)
        else:
            used_cnt = len(features)

        # 進行 5 折晶圓完全隔離驗證
        fold_maes = []
        for tr_idx, va_idx in gkf.split(X, y, wafers):
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

            v_tr, v_va = y_tr.dropna().index, y_va.dropna().index

            params = BEST_HYPERPARAMS[s_idx].copy()
            params.update({"random_state": 42, "n_jobs": -1, "verbosity": -1})
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr.loc[v_tr], y_tr.loc[v_tr])

            preds = model.predict(X_va.loc[v_va])
            fold_maes.append(mean_absolute_error(y_va.loc[v_va], preds))

        stage_mae = np.mean(fold_maes)
        all_stage_maes.append(stage_mae)
        print(
            f"  Sensor {s_idx} ({target_col:<22}) | Feats: {used_cnt:>2} | 5-Fold MAE: {stage_mae:.4f} °C"
        )

    print("=" * 75)
    print(f"  >>> 全 6 階段總平均 MAE: {np.mean(all_stage_maes):.4f} °C\n")


# ==============================================================================
# 4. 全量 25 片生產重訓與 Artifacts 導出模組 (Production Export Mode)
# ==============================================================================


def train_production_models_all25(data_dir: str, output_dir: str):
    """使用全部 25 片晶圓重訓最優結構，導出 .pkl 與 schema.json 供 Edge 使用"""
    os.makedirs(output_dir, exist_ok=True)
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        print(f"錯誤：在目錄 '{data_dir}' 找不到 *RawResult.csv 檔案！")
        return

    print("\n" + "=" * 75)
    print(
        f" 正式生產部署訓練：使用全量 {len(filepaths)} 片晶圓 (2,000 顆晶粒)"
    )
    print("=" * 75)

    dfs = [read_wafer_csv(fp) for fp in filepaths]
    df_full = pd.concat(dfs, ignore_index=True)
    raw_header = list(pd.read_csv(filepaths[0], nrows=1).columns)

    feature_schemas: Dict[int, List[str]] = {}

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = clean_col(sensor["col"])

        # 1. 取得因果合法欄位
        features = get_causal_columns(raw_header, target_col, df_full)
        for prev in SENSOR_TARGETS:
            prev_col = clean_col(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in features
                and prev_col in df_full.columns
            ):
                features.append(prev_col)

        X = df_full[features].copy()
        y = df_full[target_col].copy()
        valid_idx = y.dropna().index

        # 2. 篩選 Top-80 特徵以消除雜訊並加速推論
        if len(features) > 80:
            selector = lgb.LGBMRegressor(
                n_estimators=40, random_state=42, verbosity=-1, n_jobs=-1
            )
            selector.fit(X.loc[valid_idx], y.loc[valid_idx])
            top_feats = (
                pd.Series(selector.feature_importances_, index=features)
                .sort_values(ascending=False)
                .head(80)
                .index.tolist()
            )
            X = X[top_feats]
            feature_schemas[s_idx] = top_feats
        else:
            feature_schemas[s_idx] = features

        # 3. 載入調優參數進行全量訓練
        params = BEST_HYPERPARAMS[s_idx].copy()
        params.update({"random_state": 42, "n_jobs": -1, "verbosity": -1})
        model = lgb.LGBMRegressor(**params)
        model.fit(X.loc[valid_idx], y.loc[valid_idx])

        # 4. 保存各階段模型
        model_path = os.path.join(output_dir, f"model_sensor{s_idx}.pkl")
        with open(model_path, "wb") as f:
            pickle.dump(model, f)

        print(
            f"  [✓ 匯出成功] Sensor {s_idx} ({target_col:<22}) -> 採用特徵數: {len(feature_schemas[s_idx]):>2}"
        )

    # 5. 保存供 Edge container sample.py 讀取的特徵欄位字典
    schema_path = os.path.join(output_dir, "feature_schema.json")
    with open(schema_path, "w") as f:
        json.dump(feature_schemas, f, indent=2)

    print(
        f"\n[完成] 6 個生產模型與 feature_schema.json 已成功保存至 '{output_dir}/'！"
    )


# ==============================================================================
# 5. 主執行入口 (VS Code 直接點擊 Run 或 F5 執行)
# ==============================================================================

if __name__ == "__main__":
    # ==========================================================================
    # 執行模式切換：
    #   "VALIDATE"     : 跑 5-Fold 晶圓交叉驗證 (看 MAE 成效)
    #   "TRAIN_ALL_25" : 使用全量 25 片晶圓重訓，導出最終上機的模型檔案 (正式交付)
    #   "BOTH"         : 先跑驗證，確認無誤後自動重訓並匯出模型
    # ==========================================================================
    EXECUTION_MODE = "BOTH"

    DATA_DIR = "./Data"  # 存放 *RawResult.csv 的資料夾路徑
    OUTPUT_DIR = "./models"  # 模型輸出存放路徑

    print(f"啟動管線，當前模式: [{EXECUTION_MODE}]")

    if EXECUTION_MODE in ("VALIDATE", "BOTH"):
        run_5fold_validation(DATA_DIR)

    if EXECUTION_MODE in ("TRAIN_ALL_25", "BOTH"):
        train_production_models_all25(DATA_DIR, OUTPUT_DIR)

啟動管線，當前模式: [BOTH]

載入 25 片晶圓進行 5-Fold 跨晶圓盲測...


C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:131: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:131: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic


 5-FOLD WAFER GROUP VALIDATION RESULTS (TOP-80 PRUNED)
  Sensor 1 (100_Main.sensor1#CP   ) | Feats: 36 | 5-Fold MAE: 0.0137 °C
  Sensor 2 (120_Main.sensor2#DS0  ) | Feats: 80 | 5-Fold MAE: 0.0223 °C
  Sensor 3 (140_Main.sensor3#IO4  ) | Feats: 80 | 5-Fold MAE: 0.0275 °C
  Sensor 4 (160_Main.sensor4#IO1  ) | Feats: 80 | 5-Fold MAE: 0.0242 °C
  Sensor 5 (180_Main.sensor5#IO2  ) | Feats: 80 | 5-Fold MAE: 0.0231 °C
  Sensor 6 (200_Main.sensor6#IO3  ) | Feats: 80 | 5-Fold MAE: 0.0205 °C
  >>> 全 6 階段總平均 MAE: 0.0219 °C


 正式生產部署訓練：使用全量 25 片晶圓 (2,000 顆晶粒)


C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:125: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:131: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\724060575.py:131: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic

  [✓ 匯出成功] Sensor 1 (100_Main.sensor1#CP   ) -> 採用特徵數: 36
  [✓ 匯出成功] Sensor 2 (120_Main.sensor2#DS0  ) -> 採用特徵數: 80
  [✓ 匯出成功] Sensor 3 (140_Main.sensor3#IO4  ) -> 採用特徵數: 80
  [✓ 匯出成功] Sensor 4 (160_Main.sensor4#IO1  ) -> 採用特徵數: 80
  [✓ 匯出成功] Sensor 5 (180_Main.sensor5#IO2  ) -> 採用特徵數: 80
  [✓ 匯出成功] Sensor 6 (200_Main.sensor6#IO3  ) -> 採用特徵數: 80

[完成] 6 個生產模型與 feature_schema.json 已成功保存至 './models/'！


In [5]:
import glob
import os
import re
from typing import Dict, List
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

# ==============================================================================
# 1. 硬體時序與測試項定義
# ==============================================================================

SENSOR_TARGETS = [
    {"index": 1, "col": "100_Main.sensor1#CP", "name": "sensor1"},
    {"index": 2, "col": "120_Main.sensor2#DS0", "name": "sensor2"},
    {"index": 3, "col": "140_Main.sensor3#IO4", "name": "sensor3"},
    {"index": 4, "col": "160_Main.sensor4#IO1", "name": "sensor4"},
    {"index": 5, "col": "180_Main.sensor5#IO2", "name": "sensor5"},
    {"index": 6, "col": "200_Main.sensor6#IO3", "name": "sensor6"},
]

EXCLUDED_COLS = ["PID", "Lot", "Wafer", "PF", "SBin", "HBin", "Test Time"]


def clean_col(name: str) -> str:
    """清理 LightGBM C++ 不支援的 JSON 字元"""
    return re.sub(r"[\[\]\{\}:\",]", "_", name)


def read_wafer_csv(filepath: str) -> pd.DataFrame:
    """讀取單片晶圓 CSV，建立 Touchdown 推進、Log IDDQ 與跨 Site 基準"""
    df = pd.read_csv(filepath, skiprows=[1, 2, 3, 4], low_memory=False)
    df.columns = [clean_col(c) for c in df.columns]

    numeric_cols = [c for c in df.columns if c not in ["Lot", "Wafer"]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 1. 承載盤 Touchdown 推進進度
    if "PID" in df.columns:
        df["Touchdown_Idx"] = (df["PID"] - 1) // 4
    else:
        df["Touchdown_Idx"] = df.index // 4

    # 2. 靜態漏電流對數物理特徵 (Log Subthreshold Leakage)
    for ic in [c for c in df.columns if "IDDQ" in c]:
        df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))

    # 3. 同 Touchdown 跨 4 個 Site 的基準中心偏置
    key_iddq = clean_col("80000_Main.IDDQ_flow.IDDQ_A1#IO1")
    if key_iddq in df.columns:
        df["Touchdown_Mean_IDDQ_A1"] = df.groupby("Touchdown_Idx")[
            key_iddq
        ].transform("mean")
        df["Site_Relative_IDDQ_A1"] = df[key_iddq] - df["Touchdown_Mean_IDDQ_A1"]

    return df


def get_causal_columns(
    raw_header: List[str], target_col: str, df: pd.DataFrame
) -> List[str]:
    """嚴格防止未來數據外洩：只提取在該 Sensor 呼叫前已經執行的測試項"""
    clean_raw = [clean_col(c) for c in raw_header]
    target_idx = clean_raw.index(target_col)

    pre_tests = clean_raw[10:target_idx]
    sensor_names = [clean_col(s["col"]) for s in SENSOR_TARGETS]

    base_features = [
        c
        for c in pre_tests
        if c not in EXCLUDED_COLS and c not in sensor_names and c in df.columns
    ]

    context_features = [
        "Site",
        "X",
        "Y",
        "Touchdown_Idx",
        "Touchdown_Mean_IDDQ_A1",
        "Site_Relative_IDDQ_A1",
    ]
    log_iddq = [c for c in df.columns if c.startswith("log_IDDQ")]

    all_candidates = [
        c
        for c in (context_features + log_iddq + base_features)
        if c in df.columns
    ]
    return list(dict.fromkeys(all_candidates))


# ==============================================================================
# 2. 跨晶圓超參數自動搜尋核心
# ==============================================================================


def tune_all_sensors(data_dir: str = "./Data", n_iter: int = 35):
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*RawResult.csv")))
    if not filepaths:
        print(f"錯誤：在目錄 '{data_dir}' 找不到 *RawResult.csv 檔案！")
        return

    print(f"\n載入 {len(filepaths)} 片晶圓進行新架構超參數自動搜尋 (n_iter={n_iter})...")
    dfs = [read_wafer_csv(fp) for fp in filepaths]
    df_all = pd.concat(dfs, ignore_index=True)
    raw_header = list(pd.read_csv(filepaths[0], nrows=1).columns)

    wafers = df_all["Wafer"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    # 針對小樣本 (2,000 顆 Die) 與 80 個特徵設計的精準參數搜尋網格
    param_grid = {
        "num_leaves": [7, 10, 14, 18, 24],
        "max_depth": [3, 4, 5, 6],
        "min_child_samples": [10, 15, 20, 30],
        "learning_rate": [0.04, 0.07, 0.10, 0.14],
        "n_estimators": [60, 90, 120, 150],
        "colsample_bytree": [0.60, 0.70, 0.80, 0.90, 1.0],
        "subsample": [0.8, 0.9, 1.0],
        "reg_alpha": [0.0, 0.02, 0.05, 0.2, 0.5],
        "reg_lambda": [0.5, 1.0, 2.0, 4.0, 6.0],
        "objective": ["regression", "huber"],
    }

    best_hyperparams_result: Dict[int, dict] = {}
    summary_records = []

    print("\n" + "=" * 80)
    print(" STARTING HYPERPARAMETER TUNING ACROSS SENSORS (TOP-80 PIPELINE)")
    print("=" * 80)

    for sensor in SENSOR_TARGETS:
        s_idx = sensor["index"]
        target_col = clean_col(sensor["col"])

        # 1. 取得因果時序欄位
        features = get_causal_columns(raw_header, target_col, df_all)
        for prev in SENSOR_TARGETS:
            prev_col = clean_col(prev["col"])
            if (
                prev["index"] < s_idx
                and prev_col not in features
                and prev_col in df_all.columns
            ):
                features.append(prev_col)

        X = df_all[features].copy()
        y = df_all[target_col].copy()
        valid_idx = y.dropna().index

        # 2. 完全對齊管線：特徵若 > 80 維，先以 LightGBM 特徵重要度篩選出 Top-80
        if len(features) > 80:
            quick_selector = lgb.LGBMRegressor(
                n_estimators=40, random_state=42, verbosity=-1, n_jobs=-1
            )
            quick_selector.fit(X.loc[valid_idx], y.loc[valid_idx])
            top_feats = (
                pd.Series(quick_selector.feature_importances_, index=features)
                .sort_values(ascending=False)
                .head(80)
                .index.tolist()
            )
            X = X[top_feats]
            used_cnt = 80
        else:
            used_cnt = len(features)

        # 3. 執行 5-Fold Leave-Wafer-Out 隨機搜尋
        base_estimator = lgb.LGBMRegressor(
            random_state=42, verbosity=-1, n_jobs=2
        )

        searcher = RandomizedSearchCV(
            estimator=base_estimator,
            param_distributions=param_grid,
            n_iter=n_iter,
            scoring="neg_mean_absolute_error",
            cv=gkf.split(X.loc[valid_idx], y.loc[valid_idx], wafers[valid_idx]),
            random_state=42,
            n_jobs=-1,
            verbose=0,
        )

        print(
            f"\n>>> 正在搜尋 Sensor {s_idx} ({target_col}) [候選特徵數: {used_cnt}] ..."
        )
        searcher.fit(X.loc[valid_idx], y.loc[valid_idx])

        best_mae = -searcher.best_score_
        best_p = searcher.best_params_
        best_hyperparams_result[s_idx] = best_p

        print(f"    [最優收斂] 5-Fold 跨晶圓 MAE: {best_mae:.4f} °C")
        print(
            f"    [關鍵參數] objective={best_p['objective']}, lr={best_p['learning_rate']}, trees={best_p['n_estimators']}, depth={best_p['max_depth']}, leaves={best_p['num_leaves']}"
        )

        summary_records.append(
            {
                "Sensor": s_idx,
                "Target": target_col,
                "Features": used_cnt,
                "Tuned_MAE": best_mae,
            }
        )

    # ==========================================================================
    # 3. 輸出總結與一鍵複製代碼
    # ==========================================================================
    print("\n" + "=" * 80)
    print(" 調優總結清單 (TUNING SUMMARY)")
    print("=" * 80)
    df_res = pd.DataFrame(summary_records)
    print(df_res.to_string(index=False))
    print(
        f"\n>>> 6 個感測器調優後預期平均 MAE: {df_res['Tuned_MAE'].mean():.4f} °C"
    )

    print("\n" + "#" * 80)
    print("# 請將以下字典整段複製，直接替換主程式中的 BEST_HYPERPARAMS：")
    print("#" * 80)
    print("BEST_HYPERPARAMS = {")
    for s_idx, p in best_hyperparams_result.items():
        print(f"    {s_idx}: {{")
        for k, v in p.items():
            if isinstance(v, str):
                print(f'        "{k}": "{v}",')
            else:
                print(f'        "{k}": {v},')
        print("    },")
    print("}")
    print("#" * 80 + "\n")


# ==============================================================================
# 3. 執行入口
# ==============================================================================
if __name__ == "__main__":
    DATA_DIRECTORY = "./Data"  # 指向你的 25 片 CSV 資料夾路徑
    N_ITERATIONS = 40  # 每個感測器抽樣評估的參數組合數 (35~40 組即非常充分)

    tune_all_sensors(data_dir=DATA_DIRECTORY, n_iter=N_ITERATIONS)


載入 25 片晶圓進行新架構超參數自動搜尋 (n_iter=40)...


C:\Users\morga\AppData\Local\Temp\ipykernel_40944\1771382282.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Touchdown_Idx"] = (df["PID"] - 1) // 4
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\1771382282.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"log_{ic}"] = np.log(np.clip(df[ic].values, a_min=1e-3, a_max=None))
C:\Users\morga\AppData\Local\Temp\ipykernel_40944\1771382282.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whic


 STARTING HYPERPARAMETER TUNING ACROSS SENSORS (TOP-80 PIPELINE)

>>> 正在搜尋 Sensor 1 (100_Main.sensor1#CP) [候選特徵數: 36] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0137 °C
    [關鍵參數] objective=regression, lr=0.1, trees=150, depth=4, leaves=7

>>> 正在搜尋 Sensor 2 (120_Main.sensor2#DS0) [候選特徵數: 80] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0223 °C
    [關鍵參數] objective=huber, lr=0.1, trees=150, depth=6, leaves=7

>>> 正在搜尋 Sensor 3 (140_Main.sensor3#IO4) [候選特徵數: 80] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0275 °C
    [關鍵參數] objective=regression, lr=0.14, trees=90, depth=3, leaves=7

>>> 正在搜尋 Sensor 4 (160_Main.sensor4#IO1) [候選特徵數: 80] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0242 °C
    [關鍵參數] objective=regression, lr=0.1, trees=150, depth=6, leaves=10

>>> 正在搜尋 Sensor 5 (180_Main.sensor5#IO2) [候選特徵數: 80] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0231 °C
    [關鍵參數] objective=regression, lr=0.1, trees=150, depth=6, leaves=10

>>> 正在搜尋 Sensor 6 (200_Main.sensor6#IO3) [候選特徵數: 80] ...
    [最優收斂] 5-Fold 跨晶圓 MAE: 0.0205 °C
    [關鍵參數] objecti